# Practical Machine Learning: From Basics to Agricultural Applications

**A hands-on Jupyter Notebook module**

This notebook takes you from the fundamentals of machine learning to a complete practical workflow:

**Python → Data → EDA → Preprocessing → Regression → Classification → Model Comparison → Cross-Validation → GridSearchCV → Pipelines → Feature Engineering → Feature Importance → Clustering → PCA → Time Series → Agricultural ML Project**

### Learning objectives
By the end of this notebook, you should be able to:
- prepare and explore datasets for ML
- distinguish regression, classification and clustering problems
- preprocess numerical and categorical data
- train and evaluate several ML algorithms
- use cross-validation and GridSearchCV
- build leakage-safe pipelines
- interpret feature importance
- apply ML to crop-yield and climate-risk problems

### Recommended references
- Scikit-learn: https://scikit-learn.org/stable/user_guide.html
- Google Machine Learning Crash Course: https://developers.google.com/machine-learning/crash-course
- Pandas: https://pandas.pydata.org/docs/
- NumPy: https://numpy.org/doc/stable/
- Matplotlib: https://matplotlib.org/stable/users/index.html
- Seaborn: https://seaborn.pydata.org/
- Kaggle Learn: https://www.kaggle.com/learn
- SHAP: https://shap.readthedocs.io/


## 1. Setup and Libraries

We begin with the core Python libraries used throughout the module.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
import sklearn
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    KFold,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Regression
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingRegressor,
    GradientBoostingClassifier
)

# Classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Unsupervised learning
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

## 2. What Is Machine Learning?

Machine learning allows computers to learn patterns from data and use those patterns to make predictions or decisions.

### Main types

| Type | Goal | Example |
|---|---|---|
| Supervised learning | Learn from labeled data | Crop yield prediction |
| Classification | Predict categories | Climate risk: Low/High |
| Regression | Predict a number | Yield in t/ha |
| Unsupervised learning | Find patterns without labels | Farmer/soil clustering |
| Dimensionality reduction | Reduce variables while retaining information | PCA |

### General ML workflow

**Define problem → Collect data → EDA → Clean → Preprocess → Feature engineering → Split data → Train models → Cross-validation → Tune → Evaluate → Interpret → Deploy/Use**

## 3. NumPy Basics

In [ ]:
x = np.array([10, 20, 30, 40, 50])

print("Array:", x)
print("Mean:", np.mean(x))
print("Median:", np.median(x))
print("Standard deviation:", np.std(x))
print("Minimum:", np.min(x))
print("Maximum:", np.max(x))

## 4. Pandas Basics

Create a small agricultural dataset to practice DataFrame operations.

In [ ]:
data = {
    "Temperature": [25, 27, 30, 32, 29, 28, 31, 26],
    "Rainfall": [120, 100, 80, 60, 90, 110, 70, 130],
    "Humidity": [70, 68, 60, 55, 65, 72, 58, 75],
    "Soil_Moisture": [30, 28, 22, 18, 25, 29, 20, 32],
    "Fertilizer": [100, 110, 90, 80, 95, 105, 85, 115],
    "Yield": [4.2, 4.5, 4.0, 3.5, 4.1, 4.4, 3.7, 4.6]
}

ag_df = pd.DataFrame(data)
ag_df

In [ ]:
# Basic inspection
print("Shape:", ag_df.shape)
print("\nData types:")
print(ag_df.dtypes)

print("\nSummary statistics:")
display(ag_df.describe())

print("\nMissing values:")
print(ag_df.isnull().sum())

print("\nDuplicate rows:", ag_df.duplicated().sum())

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand distributions, relationships, missing values and possible outliers before modeling.

In [ ]:
# Histogram
sns.histplot(data=ag_df, x="Yield", kde=True)
plt.title("Distribution of Crop Yield")
plt.show()

In [ ]:
# Boxplot
sns.boxplot(data=ag_df, x="Yield")
plt.title("Crop Yield Boxplot")
plt.show()

In [ ]:
# Scatter plot
sns.scatterplot(data=ag_df, x="Rainfall", y="Yield")
plt.title("Rainfall vs Crop Yield")
plt.show()

In [ ]:
# Correlation matrix
corr = ag_df.corr(numeric_only=True)

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

## 6. Regression: Predict Crop Yield

Regression predicts a continuous numerical target.

Here, the target is **Yield**.

In [ ]:
X = ag_df.drop("Yield", axis=1)
y = ag_df["Yield"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

In [ ]:
# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

### Regression metrics

- **MAE:** average absolute prediction error.
- **RMSE:** gives greater weight to larger errors.
- **R²:** proportion of target variation explained by the model.

## 7. Classification: Titanic Survival

We now use the built-in Seaborn Titanic dataset to learn classification.

In [ ]:
titanic = sns.load_dataset("titanic")

titanic.head()

In [ ]:
# Select features and target
X = titanic[
    ["pclass", "sex", "age", "sibsp", "parch", "fare"]
].copy()

y = titanic["survived"]

# Missing age values
X["age"] = X["age"].fillna(X["age"].median())

# Encode categorical variable
X = pd.get_dummies(
    X,
    columns=["sex"],
    drop_first=True
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X.head())

In [ ]:
# Logistic Regression
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred))

## 8. Feature Scaling

Algorithms such as KNN and SVM are sensitive to the scale of numerical variables. StandardScaler transforms features approximately to mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)

## 9. KNN, Decision Tree, Random Forest and SVM

In [ ]:
models = {
    "KNN": (
        KNeighborsClassifier(n_neighbors=5),
        X_train_scaled,
        X_test_scaled
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        X_train,
        X_test
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, random_state=42),
        X_train,
        X_test
    ),
    "SVM": (
        SVC(),
        X_train_scaled,
        X_test_scaled
    )
}

comparison = []

for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)

    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

comparison_df = pd.DataFrame(comparison).sort_values(
    "F1",
    ascending=False
)

comparison_df

## 10. Cross-Validation

A single train/test split can give an unstable estimate. Cross-validation repeatedly trains and validates the model on different subsets.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

cv_scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("CV scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())
print("CV standard deviation:", cv_scores.std())

## 11. GridSearchCV

Hyperparameter tuning searches for better model settings.

The code below reports:
1. best parameters
2. best cross-validation score
3. best model
4. final test accuracy

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Score:")
print(grid_search.best_score_)

print("\nBest Model:")
print(grid_search.best_estimator_)

In [ ]:
best_model = grid_search.best_estimator_

best_pred = best_model.predict(X_test)

print("Final Test Accuracy:",
      accuracy_score(y_test, best_pred))

print("\nClassification Report:")
print(classification_report(y_test, best_pred))

## 12. Pipelines

A pipeline keeps preprocessing and modeling together and helps prevent data leakage.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
pipeline_pred = pipeline.predict(X_test)

print("Pipeline accuracy:",
      accuracy_score(y_test, pipeline_pred))

## 13. Feature Engineering

Domain knowledge can create useful predictors.

Examples for agriculture:
- water-use efficiency
- growing degree days
- heat-stress days
- rainfall accumulation
- fertilizer-use efficiency

In [ ]:
# Create a small feature-engineering example
feature_df = ag_df.copy()

feature_df["Yield_per_Fertilizer"] = (
    feature_df["Yield"] / feature_df["Fertilizer"]
)

feature_df["Water_Stress_Index"] = (
    feature_df["Temperature"] /
    (feature_df["Soil_Moisture"] + 1)
)

feature_df.head()

## 14. Feature Importance

Tree-based models can provide a first view of which variables contributed most to prediction.

In [ ]:
# Fit a Random Forest to the Titanic data
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train, y_train)

importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importance.plot(kind="bar")
plt.title("Random Forest Feature Importance")
plt.ylabel("Importance")
plt.show()

display(importance.to_frame("Importance"))

## 15. Unsupervised Learning: K-Means Clustering

Clustering finds groups without a target variable.

Agricultural examples:
- farmer typologies
- soil groups
- production systems
- climate zones

In [ ]:
cluster_data = ag_df[
    ["Temperature", "Rainfall", "Humidity", "Soil_Moisture", "Fertilizer"]
].copy()

cluster_scaler = StandardScaler()
cluster_scaled = cluster_scaler.fit_transform(cluster_data)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(cluster_scaled)

cluster_data["Cluster"] = clusters

cluster_data.head()

In [ ]:
sns.scatterplot(
    data=cluster_data,
    x="Rainfall",
    y="Soil_Moisture",
    hue="Cluster"
)

plt.title("Agricultural Data Clusters")
plt.show()

## 16. Principal Component Analysis (PCA)

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(cluster_scaled)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

print("Total explained variance:",
      pca.explained_variance_ratio_.sum())

In [ ]:
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1]
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Agricultural Variables")
plt.show()

## 17. Time-Series Feature Engineering

For agricultural and climate forecasting, observations must remain in chronological order.

Examples:
- lagged rainfall
- rolling temperature
- cumulative rainfall
- heat-stress days

**Do not randomly shuffle time-series data when evaluating forecasting performance.**

In [ ]:
# Create a synthetic daily weather dataset
dates = pd.date_range(
    start="2025-01-01",
    periods=120,
    freq="D"
)

rng = np.random.default_rng(42)

weather = pd.DataFrame({
    "Date": dates,
    "Temperature": 25 + 4*np.sin(np.arange(120)/15) + rng.normal(0, 1, 120),
    "Rainfall": np.maximum(
        0,
        rng.gamma(shape=1.5, scale=5, size=120)
    )
})

weather["Rainfall_Lag1"] = weather["Rainfall"].shift(1)
weather["Rainfall_7day_Mean"] = (
    weather["Rainfall"].rolling(7).mean()
)

weather = weather.dropna()

weather.head()

In [ ]:
plt.plot(
    weather["Date"],
    weather["Rainfall_7day_Mean"]
)

plt.xlabel("Date")
plt.ylabel("7-day Mean Rainfall")
plt.title("Rolling Rainfall Feature")
plt.xticks(rotation=45)
plt.show()

# 18. Final Practical Project: Crop Yield Prediction

This section is the template for a real agricultural ML project.

### Recommended input variables

**Weather**
- temperature
- rainfall
- humidity
- solar radiation

**Soil**
- pH
- organic carbon
- nitrogen
- phosphorus
- potassium
- soil moisture

**Management**
- variety
- fertilizer
- irrigation
- sowing date
- plant density

**Target**
- crop yield (e.g., t/ha)

### Project workflow

**Problem definition → Data collection → Data dictionary → Data quality → EDA → Preprocessing → Feature engineering → Baseline → Model comparison → Cross-validation → GridSearchCV → Final test → Feature importance/SHAP → Interpretation → Reporting**

In [ ]:
# TEMPLATE FOR YOUR REAL DATASET
# Replace "your_agriculture_data.csv" with your actual file.

# df = pd.read_csv("your_agriculture_data.csv")

# Example:
# target = "Yield"
# X = df.drop(columns=[target])
# y = df[target]

# print(df.shape)
# display(df.head())
# display(df.describe(include="all"))

## 19. A More Robust Agricultural Pipeline

For a real dataset containing both numerical and categorical variables, use `ColumnTransformer` and `Pipeline`.

The template below is designed for a mixed agricultural dataset.

In [ ]:
# Example template
# Uncomment and adapt after loading your real dataset.

# numerical_features = [
#     "Temperature",
#     "Rainfall",
#     "Humidity",
#     "Soil_Moisture",
#     "Fertilizer"
# ]

# categorical_features = [
#     "Variety",
#     "Irrigation"
# ]

# preprocessor = ColumnTransformer([
#     ("num", StandardScaler(), numerical_features),
#     ("cat", OneHotEncoder(handle_unknown="ignore"),
#      categorical_features)
# ])

# model_pipeline = Pipeline([
#     ("preprocessor", preprocessor),
#     ("model", RandomForestRegressor(random_state=42))
# ])

# model_pipeline.fit(X_train, y_train)
# predictions = model_pipeline.predict(X_test)

## 20. Recommended Model Selection

### Regression
Start with:
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor
4. XGBoost/LightGBM as advanced models

Evaluate using:
- MAE
- RMSE
- R²

### Classification
Start with:
1. Logistic Regression
2. KNN
3. Decision Tree
4. Random Forest
5. SVM
6. Gradient Boosting
7. XGBoost/LightGBM as advanced models

Evaluate using:
- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- Confusion matrix

### Unsupervised learning
Start with:
- K-Means
- Hierarchical clustering
- PCA

# 21. Exercises

### Beginner
1. Load the Iris dataset.
2. Inspect its shape and variables.
3. Calculate summary statistics.
4. Create histograms and scatter plots.
5. Identify missing values.

### Intermediate
6. Build a Logistic Regression classifier.
7. Build KNN, Decision Tree and Random Forest models.
8. Compare accuracy, precision, recall and F1.
9. Apply 5-fold cross-validation.
10. Use GridSearchCV to optimize Random Forest.

### Advanced
11. Build a complete preprocessing Pipeline.
12. Perform feature engineering.
13. Interpret feature importance.
14. Apply K-Means clustering.
15. Apply PCA.

### Research project
16. Obtain a real crop-yield dataset.
17. Combine weather, soil and management variables.
18. Build at least four ML models.
19. Compare cross-validation performance.
20. Tune the best model.
21. Evaluate on an independent test set.
22. Explain the most important predictors.
23. Document limitations and potential data leakage.
24. Prepare a scientific report.

# 22. Reference Resources

### Core ML
- Scikit-learn User Guide: https://scikit-learn.org/stable/user_guide.html
- Scikit-learn Cross-Validation: https://scikit-learn.org/stable/modules/cross_validation.html
- Scikit-learn Pipelines: https://scikit-learn.org/stable/modules/compose.html
- Google ML Crash Course: https://developers.google.com/machine-learning/crash-course

### Python/Data
- NumPy: https://numpy.org/doc/stable/
- Pandas: https://pandas.pydata.org/docs/
- Matplotlib: https://matplotlib.org/stable/users/index.html
- Seaborn: https://seaborn.pydata.org/

### Practice
- Kaggle Learn: https://www.kaggle.com/learn
- UCI Machine Learning Repository: https://archive.ics.uci.edu/

### Explainable ML
- SHAP: https://shap.readthedocs.io/

## Key principle

Do not judge a model only by its accuracy. A good ML project should also demonstrate:

**data quality + correct preprocessing + leakage control + appropriate validation + model comparison + interpretation + domain relevance + reproducibility.**
